<a href="https://colab.research.google.com/github/Hemant10HM/ANNDL-LAB_24mcs004/blob/main/ANN__LAB_9_GNN_Node_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Showing implementation of GNN Node classification on CORA dataset

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader


In [ ]:
# Load the Cora dataset which is good for node classification
dataset = Planetoid(root='data/Planetoid', name='Cora')
data = dataset[0]


Processing...
Done!


In [ ]:

def manual_split(data, num_train=140, num_val=500):
    torch.manual_seed(42)
    perm = torch.randperm(data.num_nodes)
    data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    data.val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    data.test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

    data.train_mask[perm[:num_train]] = True
    data.val_mask[perm[num_train:num_train + num_val]] = True
    data.test_mask[perm[num_train + num_val:]] = True
    return data

data = manual_split(data)

In [ ]:
#A 2-layer GCN
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x


In [ ]:
# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(dataset.num_node_features, 16, dataset.num_classes).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Training loop
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = pred[mask] == data.y[mask]
        accs.append(int(correct.sum()) / int(mask.sum()))
    return accs

In [ ]:
# Run training
for epoch in range(1, 501):
    loss = train()
    train_acc, val_acc, test_acc = test()
    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d}, Loss: {loss:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}')


Epoch 010, Loss: 0.0230, Val: 0.7840, Test: 0.7959
Epoch 020, Loss: 0.0219, Val: 0.7900, Test: 0.7950
Epoch 030, Loss: 0.0245, Val: 0.7900, Test: 0.7969
Epoch 040, Loss: 0.0243, Val: 0.8000, Test: 0.7979
Epoch 050, Loss: 0.0258, Val: 0.8000, Test: 0.7940
Epoch 060, Loss: 0.0211, Val: 0.7980, Test: 0.7935
Epoch 070, Loss: 0.0328, Val: 0.7800, Test: 0.7926
Epoch 080, Loss: 0.0196, Val: 0.7940, Test: 0.7950
Epoch 090, Loss: 0.0293, Val: 0.7940, Test: 0.7930
Epoch 100, Loss: 0.0275, Val: 0.7980, Test: 0.7930
Epoch 110, Loss: 0.0476, Val: 0.7980, Test: 0.7892
Epoch 120, Loss: 0.0279, Val: 0.7940, Test: 0.7935
Epoch 130, Loss: 0.0185, Val: 0.7960, Test: 0.7921
Epoch 140, Loss: 0.0176, Val: 0.7960, Test: 0.8003
Epoch 150, Loss: 0.0152, Val: 0.7960, Test: 0.8003
Epoch 160, Loss: 0.0186, Val: 0.8020, Test: 0.7979
Epoch 170, Loss: 0.0355, Val: 0.7980, Test: 0.7901
Epoch 180, Loss: 0.0271, Val: 0.7940, Test: 0.7930
Epoch 190, Loss: 0.0250, Val: 0.8080, Test: 0.7921
Epoch 200, Loss: 0.0218, Val: 0